python notebook for models training in google colab

In [ ]:
import tensorflow as tf
from keras.api.models import Sequential
from keras.api.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, RandomFlip, RandomRotation, RandomZoom,
    Rescaling, SeparableConv2D, BatchNormalization, Activation, GlobalAveragePooling2D)
from keras.api.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.api.regularizers import l2
from keras.api.utils import image_dataset_from_directory
import os
import matplotlib.pyplot as plt

In [ ]:
def prepocess_face_dataset(input_shape, batch_size):

    train_dir = os.path.join('data\face\fer-2013\train')
    val_dir = os.path.join('data\face\fer-2013\test')

    train_dataset = image_dataset_from_directory(
            directory=train_dir,
            label_mode='categorical',
            subset='training',
            seed=123,
            validation_split=0.15,
            image_size=input_shape[:2],
            color_mode='grayscale',
            batch_size=batch_size
    )
    
    val_dataset = image_dataset_from_directory(
        directory=val_dir,
        label_mode='categorical',
        subset='validation',
        seed=123,
        validation_split=0.15,
        image_size=input_shape[:2],
        color_mode='grayscale',
        batch_size=batch_size
    )

    train_dataset = preprocess_data_images(dataset=train_dataset, type_dataset='train', batch_size=batch_size, augment=True)
    val_dataset = preprocess_data_images(dataset=val_dataset, type_dataset='val', batch_size=batch_size, augment=True)

    return train_dataset, val_dataset

def preprocess_data_images(dataset, type_dataset: str, batch_size: int, augment=False):
    """metodo per applicare pre-elaborazione (normalization, rescaling, augmentation, shuffle, prefetch)
    direttamente sui dati prima di essere data in input al modello
    """
    
    # Add Rescaling layer to normalize pixel values
    normalization_layer = Rescaling(1./255)
    dataset = dataset.map(lambda x, y: (normalization_layer(x), y))

    if augment:
        # applicazione aumento dei dati
        data_augmentation = Sequential([
        RandomFlip("horizontal"),
        RandomRotation(0.1),
        RandomZoom(0.1)
        ])

        dataset = dataset.map(lambda x, y: (data_augmentation(x, training=True), y))

    # Batch all datasets.
    dataset = dataset.batch(batch_size)

    # cache mantiene le immagini in memoria dopo che sono state caricate dal disco durante la prima epoca. Ciò garantirà che il set di dati non diventi un collo di bottiglia durante l'addestramento del modello
    # prefetch sovrappone alla preelaborazione dei dati e all'esecuzione del modello durante l'addestramento
    if type_dataset == 'train':
        dataset = dataset.cache().shuffle(1000).prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    
    if type_dataset == 'val':
        dataset = dataset.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    
    return dataset


### Simple CNN

In [ ]:
class CNNFaceModel:
    def __init__(self, num_classes, input_shape, batch_size):
        self.num_classes = num_classes
        self.input_shape = input_shape
        self.batch_size = batch_size
        self.model = self._create_model()

    def _create_model(self):

        model = Sequential()

        model.add(Input(shape=self.input_shape))
        model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Conv2D(128, kernel_size=(3, 3), activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Flatten())
        model.add(Dense(512, activation='relu'))
        model.add(Dropout(0.5))
        model.add(Dense(self.num_classes, activation='softmax'))
        
        print(f"Creating Model ...\n")
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        print(f"Model Summary : \n")
        model.summary()

        return model

    def train_face_model(self, batch_size: int, epochs: int, patience=50, verbose=1):
        
        train, validation = prepocess_face_dataset(self.input_shape)
        
        # add callbacks
        early_stop = EarlyStopping('val_loss', patience=50)
        reduce_lr = ReduceLROnPlateau('val_loss', factor=0.1, patience=int(patience/4), verbose=verbose) # Reduce learning rate when a metric has stopped improving
        trained_models_path = 'models/face/' + '_cnn'
        model_names = trained_models_path + '.{epoch:02d}-{val_acc:.2f}.hdf5'
        model_checkpoint = ModelCheckpoint(model_names, 'val_loss', verbose=1,save_best_only=True)
        callbacks = [model_checkpoint, early_stop, reduce_lr]

        self.model.fit(train, batch_size=batch_size, epochs=epochs, validation_data=validation, callbacks=callbacks)

    def plot_training_history(self):
        history = self.model.history
        plt.figure(figsize=(10, 5))
        plt.plot(history['acc'], label='Train Accuracy')
        plt.plot(history['val_acc'], label='Validation Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.show()

        plt.figure(figsize=(10, 5))
        plt.plot(history['loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()

#### train and eval

In [ ]:
face_cnn_model = CNNFaceModel(num_classes=7, input_shape=(48,48,1), batch_size=64)
face_cnn_model.train_face_model(epochs=50)

In [ ]:
face_cnn_history = face_cnn_model.history
plt.figure(figsize=(10, 5))
plt.plot(face_cnn_history['acc'], label='Train Accuracy')
plt.plot(face_cnn_history['val_acc'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(face_cnn_history['loss'], label='Train Loss')
plt.plot(face_cnn_history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### MiniXception Model

In [ ]:
class ModelMiniXception:
    def __init__(self, num_classes: int, input_shape, batch_size: int) -> None:
        self.num_classes = num_classes
        self.input_shape = input_shape
        self.batch_size = batch_size
        self.model = self._create_model()

    def _create_model(self, l2_regularization=0.01):
        
        regularization = l2(l2_regularization)
        
        model = Sequential()

        # base
        model.add(Input(shape=self.input_shape))
        model.add(Conv2D(8, (3, 3), strides=(1, 1), kernel_regularizer=regularization, use_bias=False, input_shape=self.input_shape))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(Conv2D(8, (3, 3), strides=(1, 1), kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(Activation('relu'))

        # x4 blocchi

        #1
        model.add(SeparableConv2D(16, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(SeparableConv2D(16, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((3, 3), strides=(2, 2), padding='same'))

        #2
        model.add(SeparableConv2D(32, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(SeparableConv2D(32, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((3, 3), strides=(2, 2), padding='same'))

        #3
        model.add(SeparableConv2D(64, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(SeparableConv2D(64, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((3, 3), strides=(2, 2), padding='same'))

        #4
        model.add(SeparableConv2D(128, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(Activation('relu'))
        model.add(SeparableConv2D(128, (3, 3), padding='same', kernel_regularizer=regularization, use_bias=False))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((3, 3), strides=(2, 2), padding='same'))

        #
        model.add(Conv2D(self.num_classes, (3, 3), padding='same'))
        model.add(GlobalAveragePooling2D())
        model.add(Activation('softmax', name='predictions'))

        print(f"Creating Model ...\n")
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        print(f"Model Summary : \n")
        model.summary()

        return model
    
    def train_face_model(self, epochs=100, patience=50, verbose = 1):
            
            print(f"Loading and preprocess the data ...\n")
            train, validation = prepocess_face_dataset(self.input_shape, self.batch_size)
        
            # add callbacks
            early_stop = EarlyStopping('val_loss', patience=50)
            reduce_lr = ReduceLROnPlateau('val_loss', factor=0.1, patience=int(patience/4), verbose=1) # Reduce learning rate when a metric has stopped improving
            trained_models_path = 'models/face/_mini_xception'
            model_names = trained_models_path + '.{epoch:02d}-{val_acc:.2f}.hdf5'
            model_checkpoint = ModelCheckpoint(model_names, 'val_loss', verbose=verbose, save_best_only=True)
            callbacks = [model_checkpoint, early_stop, reduce_lr]
            print(f"add callbacks ...\n")

            print(f"Start training ... \n")
            self.model.fit(train, batch_size=self.batch_size, epochs=epochs, validation_data=validation, callbacks=callbacks)

#### train e eval

In [ ]:
face_miniX_model = ModelMiniXception(num_classes=7, input_shape=(48,48,1))
face_miniX_model.train_face_model()

In [ ]:
face_miniX_history = face_miniX_model.history
plt.figure(figsize=(10, 5))
plt.plot(face_miniX_history['acc'], label='Train Accuracy')
plt.plot(face_miniX_history['val_acc'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(face_miniX_history['loss'], label='Train Loss')
plt.plot(face_miniX_history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()